# HuggingFace EARS Audio Processing Test

This notebook demonstrates how to:
1. Fetch a random EARS audio file from HuggingFace (ColorSynth/GoMRI-17)
2. Load it directly into memory (no disk storage)
3. Apply wavelet denoising (waveclean)
4. Visualize the results

**Dataset**: ColorSynth/GoMRI-17 contains ~1000 EARS .200 files from Gulf of Mexico buoys

## 1. Install and Import Required Libraries

In [ ]:
# Install huggingface_hub if not already installed
# !pip install huggingface_hub

import random
import tempfile
import os
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
import numpy as np
import matplotlib.pyplot as plt

# Import dolphain library (local)
import sys
sys.path.insert(0, '/Users/mjhaas/code/dolphain')
from ears_reader import read_ears_file, wavelet_denoise, plot_denoising_comparison

print("✓ Libraries imported successfully")

## 2. Configure HuggingFace Repository Access

In [ ]:
# HuggingFace dataset configuration
DATASET_ID = "ColorSynth/GoMRI-17"
DATA_PATH = "2017_South/BUOY200"  # Path within the dataset
FILE_EXTENSION = ".200"  # EARS file format

# Initialize HuggingFace API
api = HfApi()

print(f"✓ Configured access to dataset: {DATASET_ID}")
print(f"  Looking for files in: {DATA_PATH}")

## 3. Fetch Random Audio File from Repository

In [ ]:
# List all files in the dataset path
print("Fetching file list from HuggingFace...")
try:
    files = api.list_repo_files(repo_id=DATASET_ID, repo_type="dataset")
    
    # Filter for EARS files in the target directory
    ears_files = [
        f for f in files 
        if f.startswith(DATA_PATH) and f.endswith(FILE_EXTENSION)
    ]
    
    print(f"✓ Found {len(ears_files)} EARS files in {DATA_PATH}")
    
    if len(ears_files) == 0:
        raise ValueError(f"No {FILE_EXTENSION} files found in {DATA_PATH}")
    
    # Select a random file
    random_file = random.choice(ears_files)
    filename = Path(random_file).name
    
    print(f"\n🎲 Randomly selected file: {filename}")
    print(f"   Full path: {random_file}")
    
except Exception as e:
    print(f"❌ Error fetching file list: {e}")
    raise

## 4. Load Audio File as EARS Object

Download the file to a temporary location and load it using the EARS reader.

In [ ]:
# Create a temporary directory
temp_dir = tempfile.mkdtemp()
print(f"Created temporary directory: {temp_dir}")

try:
    # Download file from HuggingFace to temp directory
    print(f"\nDownloading {filename}...")
    local_path = hf_hub_download(
        repo_id=DATASET_ID,
        filename=random_file,
        repo_type="dataset",
        local_dir=temp_dir,
        local_dir_use_symlinks=False
    )
    
    file_size_mb = os.path.getsize(local_path) / (1024 * 1024)
    print(f"✓ Downloaded successfully ({file_size_mb:.2f} MB)")
    print(f"  Temporary location: {local_path}")
    
    # Read the EARS file
    print("\nReading EARS file...")
    ears_data = read_ears_file(local_path, normalize=True)
    
    # Display file information
    print(f"\n📊 EARS File Information:")
    print(f"   Recording start: {ears_data['time_start']}")
    print(f"   Duration: {ears_data['duration']:.2f} seconds")
    print(f"   Sampling rate: {ears_data['fs']:,} Hz")
    print(f"   Number of samples: {ears_data['n_samples']:,}")
    print(f"   Data shape: {ears_data['data'].shape}")
    print(f"   Data range: [{ears_data['data'].min():.3f}, {ears_data['data'].max():.3f}]")
    
except Exception as e:
    print(f"❌ Error loading EARS file: {e}")
    raise

## 5. Apply Waveclean (Wavelet Denoising) to Audio

Use wavelet denoising to reduce noise while preserving biological signals like dolphin whistles and clicks.

In [ ]:
# Apply wavelet denoising
print("Applying wavelet denoising...")
try:
    # Use Daubechies 20 wavelet (good for acoustic signals)
    denoised_data, threshold_used = wavelet_denoise(
        ears_data['data'],
        wavelet='db20',
        return_threshold=True,
        hard_threshold=False  # Soft thresholding for smoother results
    )
    
    print(f"✓ Denoising complete")
    print(f"   Wavelet: db20 (Daubechies 20)")
    print(f"   Threshold used: {threshold_used:.2f}")
    print(f"   Denoised data shape: {denoised_data.shape}")
    print(f"   Denoised data range: [{denoised_data.min():.3f}, {denoised_data.max():.3f}]")
    
    # Calculate noise reduction
    noise = ears_data['data'] - denoised_data
    noise_power = np.mean(noise**2)
    signal_power = np.mean(denoised_data**2)
    snr_improvement = 10 * np.log10(signal_power / noise_power)
    
    print(f"   SNR improvement: {snr_improvement:.2f} dB")
    
except Exception as e:
    print(f"❌ Error during denoising: {e}")
    raise

## 6. Plot the Cleaned Audio

Visualize both the original and denoised audio with waveforms and spectrograms.

In [ ]:
# Use the built-in comparison plot
print("Generating comparison plots...")
plot_denoising_comparison(
    ears_data,
    wavelet='db20',
    fmax=50000,  # Show up to 50 kHz (dolphins whistle 2-20 kHz, clicks up to 150 kHz)
    xlim=None,   # Show full duration
    figsize=(16, 10)
)
print("✓ Plots generated")

## 7. Plot a Zoomed View (Optional)

Zoom in to see details of dolphin vocalizations.

In [ ]:
# Create a zoomed view of an interesting section
# Let's look at the first 10 seconds
zoom_start = 0
zoom_end = min(10, ears_data['duration'])

print(f"Generating zoomed view ({zoom_start}-{zoom_end} seconds)...")
plot_denoising_comparison(
    ears_data,
    wavelet='db20',
    xlim=(zoom_start, zoom_end),
    fmax=25000,  # Focus on whistle range (2-20 kHz)
    figsize=(16, 10)
)
print("✓ Zoomed plots generated")

## 8. Cleanup Temporary Files

In [ ]:
# Clean up temporary directory
import shutil

try:
    shutil.rmtree(temp_dir)
    print(f"✓ Cleaned up temporary directory: {temp_dir}")
except Exception as e:
    print(f"⚠️  Warning: Could not clean up temp directory: {e}")

## Summary

This notebook successfully demonstrates:
- ✅ Fetching EARS files from HuggingFace without local storage
- ✅ Loading EARS binary format directly into memory
- ✅ Applying wavelet denoising to reduce noise
- ✅ Visualizing results with waveforms and spectrograms

**Next Steps for Web Portal**:
1. Convert EARS to browser-playable format (MP3/WAV)
2. Generate spectrograms server-side or client-side (Web Audio API)
3. Create interactive visualizations with zoom/pan
4. Add whistle detection overlay
5. Enable batch processing and comparison